# Entrenar sEMG-MANUS en otra PC con GPU

Este notebook se entrega **sin ejecutar**. Abre una copia del proyecto en el equipo que realizará el entrenamiento, instala las dependencias y prepara los datos siguiendo `training/manus/README.md`. No instala software, descarga datos, sube archivos ni usa servicios externos automáticamente.

El piloto aprende **8 EMG → 20 salidas continuas nativas de MANUS**. No clasifica gestos ni convierte automáticamente esas salidas en quince ángulos anatómicos calibrados. Lee `DATASET_CARD.md` antes de interpretar resultados.

Configura las rutas, ejecuta las comprobaciones y después la celda de entrenamiento. La **prueba final está en una celda independiente**: resérvala hasta cerrar el ajuste con validación. Se exige CUDA; el notebook no habilita entrenamiento por CPU.


In [ ]:
from pathlib import Path
import os

# Si Jupyter se abre fuera del proyecto, escribe aquí su ruta absoluta
# o define la variable de entorno MANUS_PROJECT_ROOT.
PROJECT_ROOT = Path(os.environ.get("MANUS_PROJECT_ROOT", Path.cwd())).expanduser().resolve()
if not (PROJECT_ROOT / "training" / "manus" / "train.py").is_file():
    candidates = [path for path in PROJECT_ROOT.parents
                  if (path / "training" / "manus" / "train.py").is_file()]
    if not candidates:
        raise FileNotFoundError("Configura PROJECT_ROOT con la raíz del proyecto.")
    PROJECT_ROOT = candidates[0]

CODE = PROJECT_ROOT / "training" / "manus"
DATA = PROJECT_ROOT / "datasets" / "semg_manus" / "prepared_v2"
OUTPUT_ROOT = CODE / "runs"
RUN_NAME = "first_run"  # Usa un nombre nuevo para otro experimento.

EPOCHS = 40
BATCH_SIZE = 256
SEED = 42
WORKERS = 0  # Compatible con Windows, Linux y Jupyter.

if not RUN_NAME or Path(RUN_NAME).name != RUN_NAME or RUN_NAME in {".", ".."}:
    raise ValueError("RUN_NAME debe ser un nombre de carpeta, sin rutas.")
RUN_DIR = OUTPUT_ROOT / RUN_NAME

print("Proyecto:", PROJECT_ROOT)
print("Datos preparados:", DATA)
print("Salida del experimento:", RUN_DIR)


## Verificar los archivos preparados

Los datos grandes no necesitan formar parte del repositorio Git. En la otra PC se pueden copiar por separado o reconstruir con los scripts del README. Esta celda comprueba su presencia; no inicia una descarga ni vuelve a preparar los datos.


In [ ]:
import json

required = [
    CODE / "train.py", CODE / "evaluate.py", CODE / "data.py", CODE / "model.py",
    CODE / "requirements-train.txt",
    DATA / "emg.npy", DATA / "targets.npy", DATA / "manifest.json",
    DATA / "normalizer.json",
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Faltan archivos. Sigue el README para preparar el proyecto:\n" + "\n".join(missing))

manifest = json.loads((DATA / "manifest.json").read_text(encoding="utf-8"))
normalizer = json.loads((DATA / "normalizer.json").read_text(encoding="utf-8"))
print("Manifiesto de preparación:")
print(json.dumps(manifest, indent=2, ensure_ascii=False)[:16000])


## Dependencias y GPU

Instala manualmente las dependencias en el entorno de Python de la otra PC siguiendo el README. PyTorch debe ser compatible con su GPU NVIDIA y controlador; la [guía oficial de PyTorch](https://pytorch.org/get-started/locally/) permite elegir la instalación apropiada.

Las siguientes celdas muestran el entorno activo y comprueban CUDA. No instalan controladores ni paquetes. Si falta una dependencia o CUDA, corrige el entorno antes de entrenar.


In [ ]:
import subprocess
import sys

print("Python activo:", sys.executable)
print("Requisitos del proyecto:", CODE / "requirements-train.txt")
print("Esta celda no instala dependencias.")


In [ ]:
import numpy as np
import torch

def require_cuda():
    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA no está disponible. Activa una GPU NVIDIA en el equipo remoto "
            "y revisa su instalación de PyTorch antes de entrenar. "
            "Este notebook no continúa por CPU."
        )
    print("PyTorch:", torch.__version__)
    print("GPU:", torch.cuda.get_device_name(0))
    print("Memoria GPU:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2), "GiB")

require_cuda()

emg = np.load(DATA / "emg.npy", mmap_mode="r", allow_pickle=False)
targets = np.load(DATA / "targets.npy", mmap_mode="r", allow_pickle=False)
if emg.ndim != 2 or emg.shape[1] != 8:
    raise ValueError(f"Se esperaba EMG [filas,8]; recibido {emg.shape}.")
if targets.ndim != 2 or targets.shape[1] != 20 or targets.shape[0] != emg.shape[0]:
    raise ValueError(f"Objetivos incompatibles: {targets.shape}, EMG: {emg.shape}.")
if emg.shape[0] < 80:
    raise ValueError("No hay suficientes filas para una ventana de 80 muestras.")

print("EMG:", emg.shape, emg.dtype)
print("Objetivos:", targets.shape, targets.dtype)
print("Las filas se agrupan por ensayo según el manifiesto; no se mezclan sus límites.")
del emg, targets

## Entrenamiento con validación

Configuración inicial: **40 épocas**, batch **256**, semilla 42. La preparación define ventanas de 80 filas y salto de 10, sin cruzar ensayos. La normalización procede únicamente de entrenamiento.

Participantes: entrenamiento **5, 6, 8, 9, 10, 11, 12, 15, 16, 18**; validación **7, 14**; prueba **3, 4, 13**. El entrenador valida el esquema de datos y elige `best.pt` con validación. Esta celda no calcula métricas de prueba.

Si la GPU se queda sin memoria, reduce `BATCH_SIZE` y usa un nuevo `RUN_NAME`; registra el cambio. No compares experimentos ajustándolos al conjunto de prueba.

In [ ]:
import shlex

require_cuda()
if RUN_DIR.exists():
    raise FileExistsError(
        f"Ya existe {RUN_DIR}. Cambia RUN_NAME para conservar el experimento anterior."
    )
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

train_command = [
    sys.executable, str(CODE / "train.py"),
    "--data", str(DATA),
    "--output", str(RUN_DIR),
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--seed", str(SEED),
    "--workers", str(WORKERS),
]
print(shlex.join(train_command))
subprocess.run(train_command, cwd=PROJECT_ROOT, check=True)

if not (RUN_DIR / "best.pt").is_file():
    raise RuntimeError("El entrenador terminó sin producir best.pt.")
print("Modelo seleccionado por validación:", RUN_DIR / "best.pt")

## Revisar entrenamiento antes de abrir prueba

Los siguientes archivos describen el ajuste con entrenamiento/validación. Conservarlos permite reproducir la selección del checkpoint. No son resultados de prueba.

In [ ]:
for filename in ("config.json", "history.json", "training_summary.json"):
    path = RUN_DIR / filename
    print("\n" + filename)
    if path.is_file():
        value = json.loads(path.read_text(encoding="utf-8"))
        print(json.dumps(value, indent=2, ensure_ascii=False)[:24000])
    else:
        print("No se generó este archivo; revisa la salida del entrenador.")

## Evaluación final de prueba — celda independiente

Ejecuta esta celda después de cerrar las decisiones con validación. Evalúa los participantes reservados **3, 4 y 13**, guarda métricas y exporta el modelo. Para evitar reemplazar una evaluación previa, la celda se detiene si `test_metrics.json` ya existe.

La exportación TorchScript recibe EMG en formato **[batch, 80, 8]**, en la escala original del dataset; incorpora la normalización aprendida con entrenamiento y devuelve **[batch, 20]** en unidades nativas MANUS. «EMG en escala original» se refiere a los valores guardados por el Myo: no acredita que sean voltajes sin filtrar ni que coincidan con la escala del EMG PRO.

In [ ]:
require_cuda()
checkpoint = RUN_DIR / "best.pt"
metrics_path = RUN_DIR / "test_metrics.json"
export_path = RUN_DIR / "model.ts"

if not checkpoint.is_file():
    raise FileNotFoundError("Falta best.pt: primero debe terminar el entrenamiento.")
if metrics_path.exists():
    raise FileExistsError(
        f"Ya existe {metrics_path}. Consérvalo; la celda siguiente permite leerlo."
    )

evaluate_command = [
    sys.executable, str(CODE / "evaluate.py"),
    "--data", str(DATA),
    "--checkpoint", str(checkpoint),
    "--output", str(metrics_path),
    "--batch-size", str(BATCH_SIZE),
    "--workers", str(WORKERS),
    "--device", "cuda",
    "--export", str(export_path),
]
print(shlex.join(evaluate_command))
subprocess.run(evaluate_command, cwd=PROJECT_ROOT, check=True)

In [ ]:
metrics_path = RUN_DIR / "test_metrics.json"
if not metrics_path.is_file():
    raise FileNotFoundError("Todavía no hay métricas de prueba.")
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print("\nInterpreta los errores como diferencias respecto a las salidas nativas MANUS.")
print("No constituyen una validación de precisión anatómica ni transferencia al EMG PRO.")

## Conservar resultados en el proyecto

Esta celda copia el normalizador y el manifiesto junto a los resultados y registra el entorno. No crea un ZIP ni sube archivos. Conserva la carpeta completa del experimento, incluidos `best.pt`, `model.ts` si se exportó, métricas, configuración e historial.

Antes de conectarlo al simulador faltan la validación del EMG PRO y el mapeo de veinte salidas, especialmente las etiquetas genéricas del pulgar. Conserva la separación entre valores solicitados por el modelo y valores alcanzados por la mano con contactos.


In [ ]:
import hashlib
import shutil
from datetime import datetime, timezone

if not (RUN_DIR / "best.pt").is_file():
    raise FileNotFoundError("No hay un checkpoint entrenado para documentar.")

provenance = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "project_path_on_training_machine": str(PROJECT_ROOT),
    "torch_version": torch.__version__,
    "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "epochs_requested": EPOCHS,
    "batch_size": BATCH_SIZE,
    "seed": SEED,
    "input_layout": ["batch", 80, 8],
    "output_channels": 20,
    "output_units": "native_MANUS",
}
for filename in ("manifest.json", "normalizer.json"):
    content = (DATA / filename).read_bytes()
    provenance[filename + "_sha256"] = hashlib.sha256(content).hexdigest()
    shutil.copy2(DATA / filename, RUN_DIR / ("prepared_" + filename))

(RUN_DIR / "notebook_provenance.json").write_text(
    json.dumps(provenance, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Resultados conservados en:", RUN_DIR)
print("Evaluación final incluida:", (RUN_DIR / "test_metrics.json").is_file())
